# 01 · Análisis exploratorio y calidad de datos

Este notebook caracteriza las particiones del dataset, valida etiquetas, analiza valores faltantes, duplicados, variables constantes y comportamiento temporal.

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scania_anomaly.config import load_config, ensure_directories
from scania_anomaly.utils.reproducibility import set_global_seed

config = load_config('../config/config.yaml')
ensure_directories(config)
set_global_seed(config['project']['seed'])

DRIVE_ROOT = Path(config['paths']['drive_root'])
RAW_DIR = Path(config['paths']['raw_dir'])
PROCESSED_DIR = Path(config['paths']['processed_dir'])
OUTPUTS_DIR = Path(config['paths']['outputs_dir'])
MODELS_DIR = Path(config['paths']['models_dir'])
METRICS_DIR = Path(config['paths']['metrics_dir'])
TABLES_DIR = Path(config['paths']['tables_dir'])
print(config['project']['name'], config['project']['version'])

from scania_anomaly.spark_session import create_spark_session
from scania_anomaly.data_loader import ScaniaDataLoader
from scania_anomaly.data_quality import DataQualityAnalyzer, save_quality_report
from scania_anomaly.labels import summarize_vehicle_labels
from scania_anomaly.temporal_analysis import time_step_gap_report, trajectory_length_report

spark = create_spark_session(config)
loader = ScaniaDataLoader.from_config(spark, config)


In [ ]:

train_op = loader.read_csv('train_operational')
val_op = loader.read_csv('validation_operational')
test_op = loader.read_csv('test_operational')
train_tte = loader.read_csv('train_tte')
val_labels = loader.read_csv('validation_labels')
test_labels = loader.read_csv('test_labels')

for name, df in [('train_op', train_op), ('validation_op', val_op), ('test_op', test_op)]:
    print(name, df.count(), len(df.columns))


In [ ]:

# Calidad de datos por partición
for name, df in [('train', train_op), ('validation', val_op), ('test', test_op)]:
    analyzer = DataQualityAnalyzer(df)
    missing = analyzer.missing_report()
    save_quality_report(missing, TABLES_DIR / f'{name}_missing_report.csv')
    print(name, 'shape:', analyzer.shape(), 'duplicados:', analyzer.duplicated_count(['vehicle_id', 'time_step']))
    display(missing.head(10))


In [ ]:

# Etiquetas a nivel vehículo
print('Validation labels:', summarize_vehicle_labels(val_labels))
print('Test labels:', summarize_vehicle_labels(test_labels))
print('Train TTE labels:', summarize_vehicle_labels(train_tte, label_col='in_study_repair'))


In [ ]:

# Análisis de irregularidad temporal y longitud de trayectorias
for name, df in [('train', train_op), ('validation', val_op), ('test', test_op)]:
    gap_report = time_step_gap_report(df)
    len_report = trajectory_length_report(df)
    gap_report.to_csv(TABLES_DIR / f'{name}_time_gap_report.csv', index=False)
    len_report.to_csv(TABLES_DIR / f'{name}_trajectory_length_report.csv', index=False)
    print('
', name)
    display(gap_report)
    display(len_report)
